In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "parron2008behavioural")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "chimpanzees-leipzig_edited.csv")
complete_path_2 = os.path.join(original_data_pathway, "Gorillas Nurnberg.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df1 = df1.assign(day='15')
df1 = df1.assign(month='5')
df1 = df1.assign(year='2006')

df2 = pd.read_csv(complete_path_2)
df2 = df2.assign(species_original='gorilla')
df2 = df2.assign(year='2006')
df2.rename(columns={"GORILLA": "name", 
                    "TRIAL":"trial_gorilla", 
                    "COMMENT":"comment_gorilla"}, inplace=True)



In [3]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"name": "ape"}, inplace=True)
    x['ape'] = x['ape'].str.rstrip()
    x['study_id']="parron2008behavioural"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

In [5]:
spe_2=[]  
for index, row in fulldf.iterrows():
    if not pd.isna(row['species']):
        spe_2.append(row['species'])
    else:
        spe_2.append(row['species_original'])
fulldf = fulldf.assign(species=spe_2)

In [6]:
replace_list = ['notes', 'comments', 'comment_gorilla']
fulldf['notes'].replace('didn�t', 'did not', inplace=True, regex=True)
for x in replace_list:
    fulldf[x].replace(',', '', inplace=True, regex=True)

# fulldf.columns
fulldf.rename(columns={"ape": "participant"}, inplace=True)


In [7]:

replace_list_3 = [['banane', 'banana'],
                  ['galet','pebble']]
                #   ['^b$', np.nan], ## add in if removing gorilla testing
                #     ['^p$', np.nan]] 
for b,c in replace_list_3:
    fulldf['left'].replace(b, c, inplace=True, regex=True)
    fulldf['right'].replace(b, c, inplace=True, regex=True)
    fulldf['choice'].replace(b, c, inplace=True, regex=True)
# fulldf.dropna(subset=['left'], inplace=True)## add in if removing gorilla testing

In [8]:
fulldf['temp_trial']=fulldf.left.str.cat(fulldf.right, sep='_')
fulldf['temp_trial'].unique()

# trial_add_list = [['banana_photo banana', '1'],
#                   ['photo banana_pebble','2'],
#                   ['photo pebble_photo banana','3_4'],
#                   ['photo banana_photo pebble','3_4'],
#                   ['pebble_photo banana','2'],
#                   ['photo banana_banana','1']]
# for x,y in trial_add_list:
#     fulldf.loc[fulldf.temp_trial == x, ['trial_new']] = y


test_list_left = ['banana_photo', 'banana', 'pebble','pebble_photo', 'photo banana','photo pebble']
output_no_2 = 1
init_var = 0
temp = []
for index, row in fulldf.iterrows():
    if pd.isna(row['left']):
        temp.append('') 
    elif row['left'] in test_list_left:
        temp.append(output_no_2) 
        output_no_2 = output_no_2+1
        if row['participant']=='sandra':
            if output_no_2 == 6: 
                output_no_2 = 1 
        elif row['participant'] != 'sandra':
            if output_no_2 == 5: ##session number can only go up to 10, so once it's 11 it needs to reset
                output_no_2 = 1 ## resets back to one
    else:
        temp.append('')
fulldf = fulldf.assign(trial_new=temp)



In [9]:
spe_3=[]  

for index, row in fulldf.iterrows():
    if row['trial_new'] == '':
        spe_3.append('training')
    else:
        spe_3.append(row['trial_new'])
fulldf = fulldf.assign(trial=spe_3)

##empty forgotten trial
fulldf.at[70, 'trial'] = ''

spe_4=[]  
for index, row in fulldf.iterrows():
    if row['species'] == 'gorilla':
        spe_4.append(row['comment_gorilla'])
    else:
        spe_4.append(row['comments'])
fulldf = fulldf.assign(comment=spe_4)

space_replace_list = ['right','left','choice','notes','comment']
for x in space_replace_list:
    fulldf[x]=fulldf[x].str.rstrip()
    fulldf[x].replace(' ', '_', inplace=True, regex=True)

In [10]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'].replace('2006-nan-nan', np.nan, inplace=True, regex=True)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])

fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [11]:
fulldf['participant'].replace('bianka', 'bianca', inplace=True, regex=True)

estimated_age = [['bianca','f','34'],
                 ['hakuna','f','10'],
                 ['lena','f','29']]
for x,y,k in estimated_age:
    fulldf.loc[fulldf.participant == x, ['sex', 'age_in_years']] = y, k

In [12]:
parron2008behavioural_standardized=fulldf[[ 'study_id','year', 'month', 'day', 
                                           'participant', 'age_in_years','sex', 'species', 'trial',
    'left', 'right', 'choice', 'notes',  'comment' ]]
comp_out_path_stand = os.path.join(out_pathway, 'parron2008behavioural_standardized.csv')
parron2008behavioural_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =parron2008behavioural_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
parron2008behavioural_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'parron2008behavioural_glossary.csv')
parron2008behavioural_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
